[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/math/klein_quadric/klein_quadric.ipynb)

# Four Lines and the Klein Quadric

This example finds the lines that meet four given lines in space. Three lines in general position have a whole family of lines meeting all three, sweeping out a hyperboloid; a fourth line crosses that hyperboloid in two points, and through each passes one line meeting all four. When the fourth line misses the hyperboloid, the two lines are still there, as a complex-conjugate pair.

A line in space is a bivector of projective geometric algebra, but most bivectors are not lines: a bivector is a line when its wedge with itself vanishes, and two lines meet when their wedge vanishes. Both are one form, `Bivector ^ Bivector`, a quadratic form on the six-dimensional space of bivectors whose zeros are the lines: the Klein quadric. Meeting four lines is four linear conditions, which leave a pencil of bivectors; along the pencil the Klein form is a quadratic, and its two zeros are the two transversals. The algebra runs on complex coefficients, so that both are there in every case.

In Plücker coordinates a line reads as six numbers on the Klein quadric in five-dimensional projective space, and in the Schubert calculus of the Grassmannian of lines the count of two reads as the intersection number of four Schubert cycles.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext, stack
from numga.algebras import PGA3D as ga
from examples.util.animation import save_figure, show_animation
from examples.math.klein_quadric import render
from examples.math.klein_quadric.core import spread

In [ ]:
# Complex coefficients, so that the two transversals are there in every case, a conjugate pair where they are not real.
mv = NumpyContext(ga, np.complex128).multivector
Scalar = ga.gatype.scalar()
Plane = ga.gatype.vector()
# Every line here is a meet of two planes, as the lines through a point that meet two lines are: an axis.
Bivector = ga.gatype.bivector()
Point = ga.gatype.antivector()
# A point joined with a line and met with others gives a plane through points: oriented within itself.
Quadric = ga.gatype((ga.gatype.antitrivector(), Point))
# Zero on one bivector twice exactly when it is a line; zero on two lines exactly when they meet.
KLEIN = Bivector ^ Bivector                                                    # [] Pseudoscalar <- (Bivector, Bivector)
# The origin, the point dual to the plane at infinity; the plane at infinity as the join of the three directions,
# whose meet with an axis is the axis's point at infinity.
ORIGIN = mv.w.dual()                                                           # [] Point
HORIZON = mv.direction([1.0, 0.0, 0.0]) & mv.direction([0.0, 1.0, 0.0]) & mv.direction([0.0, 0.0, 1.0])
# Both roots of a quadratic, one per sign.
SIGNS = np.array([1.0, -1.0])

## 1. The Klein form

The Klein form takes two bivectors to their wedge, a pseudoscalar. A meet of two planes is a line and the form vanishes on it twice; a sum of two skew lines is not a line, and the form does not. Two lines that cross pair to zero, and two that pass each other do not.

In [ ]:
# Two lines along z, through the origin and through (1, 0, 0), each the meet of two planes; and a line
# along y through (1, 0, 1), passing the first.
along_z = mv.x ^ mv.y                                                          # [] Bivector
beside = (mv.x - mv.w) ^ mv.y                                                  # [] Bivector
above = (mv.x - mv.w) ^ (mv.z - mv.w)                                          # [] Bivector

## 2. Four lines, two transversals

Each line, weighted by its Klein pairing with a bivector, is a map whose zeros are the bivectors that meet it; summed over the four lines, the map is zero only on the bivectors that meet all four. They form a pencil, the two singular vectors of zero singular value. Along the pencil the Klein form is a quadratic, and its two roots are the two transversals. Three lines are fixed through the waist of the hyperboloid x² + y² = 1 + z², and a steep fourth line is placed outside the waist and through it.

In [ ]:
# Where the three lines cross the hyperboloid's waist, by angle about z; where the fourth line crosses the plane
# z = 0 at rest and the direction it runs in; and its offsets along x: outside the waist, and through it.
waist_angles = np.array([0.3, 2.3, 4.4])
fourth_crossing, fourth_direction = [0.0, 0.4, 0.0], [0.3, -0.2, 1.0]
offsets = np.array([1.4, 0.0])

In [ ]:
def roots(b: Scalar, c: Scalar) -> Scalar:
    """The two roots of the quadratic `1 + 2 * b * t + c * t * t` along the combinations of two ends,
    as the weight of the first end against unit weight on the second, one root per sign on the last
    axis: `first * (sign * root - b) + second`, with `root * root == b * b - c`. A quadratic whose
    coefficients share a factor, such as the handedness of a signed volume, is divided by its first
    coefficient to read as this one."""
    root = (b * b - c).square_root()                                           # [...] Scalar
    return root[..., None] * SIGNS - b[..., None]                              # [..., 2] Scalar


def transversals(lines: Bivector) -> Bivector:
    """The two lines that meet all four lines on the last axis, at unit norm, on the last axis."""
    # Each line, weighted by the Klein form between it and a bivector: zero exactly on the bivectors
    # that meet all four.
    misses = (lines * KLEIN(lines).dual()).sum(axis=-1)                        # [...] Bivector <- Bivector
    # Those bivectors form a pencil, spanned by the right singular vectors of zero singular value.
    _, _, right = misses.svd()
    pencil = right[..., -2:]                                                   # [..., 2] Bivector
    # Along the pencil the Klein form is a quadratic; its two zeros are the lines of the pencil.
    first, second = pencil[..., 0], pencil[..., 1]                             # [...] Bivector
    # The Klein form of two axes is a signed volume; its ratios are ordinary numbers.
    leading = KLEIN(first, first).dual()                                       # [...] Scalar
    weights = roots(KLEIN(first, second).dual() / leading, KLEIN(second, second).dual() / leading)   # [..., 2] Scalar
    return (first[..., None] * weights + second[..., None]).normalized()       # [..., 2] Bivector


def waist_lines(angles: np.ndarray) -> Bivector:
    """Lines of the hyperboloid on which x and y squared sum to one more than z squared, through the points of its
    waist at the given angles, all leaning the same way: the meet of the planes x = 1 and y = z, turned about z."""
    return (mv.xy * (-angles / 2)).exp() >> ((mv.x - mv.w) ^ (mv.y - mv.z))


def steep(offsets: np.ndarray) -> Bivector:
    """The fourth line, moved along x by each offset: the meet of the two planes through it along x and along y."""
    base = (mv.xw * (offsets / 2)).exp() >> mv.point(fourth_crossing)        # [offsets] Point
    along = base & mv.direction(fourth_direction)                              # [offsets] Bivector
    return (along & mv.direction([1.0, 0.0, 0.0])) ^ (along & mv.direction([0.0, 1.0, 0.0]))


three = waist_lines(waist_angles)                                               # [3] Bivector
fourth = steep(offsets)                                                        # [offsets] Bivector
lines = stack([three[0].broadcast_to(fourth.shape), three[1].broadcast_to(fourth.shape),
               three[2].broadcast_to(fourth.shape), fourth], axis=-1)          # [offsets, 4] Bivector
across = transversals(lines)                                                   # [offsets, 2] Bivector

## 3. The same count, seen in space

The lines that meet three lines sweep out a quadric. As a map from points to planes it pairs to zero with its points: the plane joining a point to the first line meets the third line in a point, and the plane joining that point to the second line holds the first point exactly when a line through it meets all three. The fourth line crosses that quadric at two points, by the same quadratic as before, now along the fourth line, and through each crossing runs one of the transversals. In the animation the fourth line, orange, swings into the hyperboloid's waist and out again. While it passes outside the waist it crosses the surface at two points, and through each crossing runs a red transversal. As it nears the waist the two crossings slide together, and the two red lines with them; where the orange line touches the surface they coincide, and past that point they leave the real picture together, as a complex-conjugate pair. On the way out they return the same way. The swing slows where the crossings merge, so that the red pair is seen to close and vanish.

In [ ]:
# The lines drawn of each family; the frames of the swing and how far it reaches either way; the points the swing
# is sampled at to pace it, and the share of the line's own motion in the pace; the frame shown as a still; and the
# duration of a frame in milliseconds.
n_rulings = 36
n_frames = 96
swing = 1.5
n_path, line_share = 960, 0.6
still = 8
frame_duration = 70

In [ ]:
def ruled_quadric(first: Bivector, second: Bivector, third: Bivector) -> Quadric:
    """The quadric swept by the lines that meet three lines, as a map from points to planes that
    pairs to zero with the points on it. The plane joining a point to the first line meets the third
    line in a point; the plane joining that point to the second line holds the first point exactly
    when a line through it meets all three."""
    return ((Point & first) ^ third) & second                                  # [...] Plane <- Point


def crossings(surface: Quadric, line: Bivector) -> Point:
    """The two points where a line crosses a quadric, at unit weight, on the last axis."""
    # The line's point nearest the origin and its point at infinity span it.
    span = stack([(ORIGIN | line) ^ line, line ^ HORIZON], axis=-1)            # [..., 2] Point
    first, second = span[..., 0], span[..., 1]                                 # [...] Point
    # The quadric's map need not be symmetric: the cross term is the mean of both orders.
    cross = ((surface(first) & second) + (surface(second) & first)) / 2        # [...] Scalar
    leading = surface(first) & first                                           # [...] Scalar
    weights = roots(cross / leading, (surface(second) & second) / leading)     # [..., 2] Scalar
    crossing = first[..., None] * weights + second[..., None]                  # [..., 2] Point
    return crossing / (mv.w & crossing)                                        # [..., 2] Point


def through(points: Point, second: Bivector, third: Bivector) -> Bivector:
    """The line through each point that meets two lines: the meet of the planes joining the point to
    each line."""
    return (points & second) ^ (points & third)                                # [...] Bivector


def gap(points: Point) -> np.ndarray:
    """How far apart a line's two crossings are `[...]`: their distance while they are real, and minus it once they
    are a complex pair, so that it passes through zero where they merge."""
    apart = points[..., 0].kernel - points[..., 1].kernel                      # [..., coefficients] complex
    real = np.abs(apart.imag).max(axis=-1) < 1e-9
    return np.where(real, np.linalg.norm(apart.real, axis=-1), -np.linalg.norm(apart.imag, axis=-1))


def paced(path: np.ndarray, line_share: float) -> np.ndarray:
    """`n_frames` offsets along a closed swing, at equal steps of the gap between the fourth line's crossings and of
    `line_share` of the line's own motion: slow where the crossings merge, so the transversals are seen to close."""
    closed = np.append(path, path[0])
    gaps = gap(crossings(surface, steep(closed)))
    pace = np.concatenate([[0.0], np.cumsum(np.abs(np.diff(gaps)) + line_share * np.abs(np.diff(closed)))])
    return np.interp(np.linspace(0.0, pace[-1], n_frames, endpoint=False), pace, closed)


surface = ruled_quadric(three[0], three[1], three[2])                          # [] Plane <- Point
swung = steep(paced(swing * np.cos(np.linspace(0.0, 2 * np.pi, n_path, endpoint=False)), line_share))   # [frames] Bivector
swung_lines = stack([three[0].broadcast_to(swung.shape), three[1].broadcast_to(swung.shape),
                     three[2].broadcast_to(swung.shape), swung], axis=-1)       # [frames, 4] Bivector
swung_across = transversals(swung_lines)                                       # [frames, 2] Bivector
crossing = crossings(surface, swung)                                           # [frames, 2] Point
# The lines meeting the three, through points spread over the first; and the lines meeting three of
# those, the family of the three.
sweep = through(spread(three[0], n_rulings), three[1], three[2])               # [rulings] Bivector
family = through(spread(sweep[0], n_rulings), sweep[n_rulings // 3], sweep[2 * n_rulings // 3])   # [rulings] Bivector
rulings = stack([sweep, family])                                               # [families, rulings] Bivector
scenes = [(swung_lines[frame], rulings, swung_across[frame], crossing[frame]) for frame in range(n_frames)]
save_figure(render.draw(*scenes[still]), "klein_transversals")
show_animation(render.animate(scenes), "klein_swing", frame_duration)

In [ ]:
# checks
# The Klein form vanishes on a line and on two lines that meet, and not on a sum of skew lines or on two skew lines.
np.testing.assert_allclose(KLEIN(along_z, along_z).dual().to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose(KLEIN(along_z, beside).dual().to_array(), 0.0, atol=1e-12)
assert np.abs(KLEIN(along_z + above, along_z + above).dual().to_array()) > 0.5
assert np.abs(KLEIN(along_z, above).dual().to_array()) > 0.5
# The transversals are lines and meet all four; outside the waist both are real, through it both are complex.
np.testing.assert_allclose((lines[..., :, None] ^ across[..., None, :]).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((swung_across ^ swung_across).kernel, 0.0, atol=1e-12)
np.testing.assert_allclose((swung_lines[..., :, None] ^ swung_across[..., None, :]).kernel, 0.0, atol=1e-12)
real = np.abs(across.kernel.imag).max(axis=-1) <= 1e-6 * np.abs(across.kernel).max(axis=-1)   # [offsets, 2]
assert real[0].all() and not real[1].any()
# The crossings lie on the quadric and on a transversal each; both families lie on the quadric.
np.testing.assert_allclose((surface(crossing) & crossing).kernel, 0.0, atol=1e-10)
off = np.abs((crossing[..., :, None] & swung_across[..., None, :]).kernel).max(axis=-1).min(axis=-1)   # [frames, 2]
np.testing.assert_allclose(off, 0.0, atol=1e-10)
ends = spread(rulings, 3)                                                      # [families, rulings, 3] Point
np.testing.assert_allclose((surface(ends) & ends).kernel, 0.0, atol=1e-10)